In [1]:
# ==========================================
# Lesson 30 - Random Forest in Earth Engine
# ==========================================

import ee
import geemap

ee.Initialize(project="oc-flux")

Map = geemap.Map()

point = ee.Geometry.Point([77.2090, 28.6139])

collection = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(point)
    .filterDate("2023-01-01", "2023-12-31")
)

image = collection.median()

Map.centerObject(point, 10)

Map

Map(center=[28.613900000000005, 77.209], controls=(WidgetControl(options=['position', 'transparent_bg'], posit…

In [ ]:
Create Predictor Variables

In [2]:
ndwi = image.normalizedDifference(
    ["SR_B3", "SR_B5"]
).rename("NDWI")

mndwi = image.normalizedDifference(
    ["SR_B3", "SR_B6"]
).rename("MNDWI")

In [ ]:
Create Feature Stack

In [3]:
feature_stack = image.addBands([
    ndwi,
    mndwi
])

print(feature_stack.bandNames().getInfo())

['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'SR_QA_AEROSOL', 'ST_B10', 'ST_ATRAN', 'ST_CDIST', 'ST_DRAD', 'ST_EMIS', 'ST_EMSD', 'ST_QA', 'ST_TRAD', 'ST_URAD', 'QA_PIXEL', 'QA_RADSAT', 'NDWI', 'MNDWI']


In [ ]:
Create Example Training Points

Since we don't have real field POC data yet, we'll create a small synthetic dataset.

In [4]:
points = ee.FeatureCollection([

ee.Feature(
    ee.Geometry.Point([77.20,28.61]),
    {"POC":2.5}
),

ee.Feature(
    ee.Geometry.Point([77.22,28.63]),
    {"POC":4.2}
),

ee.Feature(
    ee.Geometry.Point([77.24,28.60]),
    {"POC":6.8}
),

ee.Feature(
    ee.Geometry.Point([77.26,28.59]),
    {"POC":7.9}
)

])

In [ ]:
Display them:

In [5]:
Map.addLayer(
    points,
    {"color":"red"},
    "Training Points"
)

Map

Map(bottom=109612.0, center=[28.613900000000005, 77.209], controls=(WidgetControl(options=['position', 'transp…

In [ ]:
Sample the Image

In [6]:
training = feature_stack.sampleRegions(

    collection=points,

    properties=["POC"],

    scale=30

)

In [ ]:
Check the size:

In [7]:
print(training.size().getInfo())

4


In [ ]:
Define Predictor Bands

In [8]:
bands = [

"SR_B2",

"SR_B3",

"SR_B4",

"SR_B5",

"NDWI",

"MNDWI"

]

In [ ]:
Train Random Forest Regression

In [9]:
rf = ee.Classifier.smileRandomForest(
    numberOfTrees=100
).setOutputMode("REGRESSION")

In [ ]:
Train it:
Congratulations!

You have trained a Random Forest model inside Earth Engine.

In [10]:
trained = rf.train(

    features=training,

    classProperty="POC",

    inputProperties=bands

)

In [ ]:
Apply the Model
Earth Engine predicts a POC value for every pixel.

In [11]:
prediction = feature_stack.classify(
    trained
)

In [ ]:
Display Prediction
You should see a colored prediction layer (remember this is based on synthetic training data, so it is only for learning).

In [12]:
Map.addLayer(

prediction,

{
    "min":2,
    "max":8,
    "palette":[
        "blue",
        "cyan",
        "green",
        "yellow",
        "red"
    ]
},

"Predicted POC"

)

Map

Map(bottom=109612.0, center=[28.613900000000005, 77.209], controls=(WidgetControl(options=['position', 'transp…

In [ ]:
Compare Input and Output

Add a true-color image:
Now switch between:

True Color
Predicted POC

Observe how the prediction layer differs from the original satellite image.

In [13]:
Map.addLayer(

image,

{
    "bands":["SR_B4","SR_B3","SR_B2"],
    "min":7000,
    "max":18000
},

"True Color"

)

Map

Map(bottom=14022.0, center=[28.05259082333986, 78.12927246093751], controls=(WidgetControl(options=['position'…

In [ ]:
Inspect Pixel Values

Click on the map using the Inspector tool (if available in geemap) or print sampled values:
This returns the average predicted POC around the selected point.

In [14]:
print(
    prediction.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=point.buffer(500),
        scale=30
    ).getInfo()
)

{'classification': 5.284999999999999}


In [ ]:
Export Prediction
This exports the prediction as a GeoTIFF to your computer.

In [18]:
geemap.ee_export_image(
    ee_object=prediction,
    filename="Predicted_POC.tif",
    scale=30,
    region=point.buffer(5000),
    file_per_band=False
)

Generating URL ...
Please wait ...
Data downloaded to C:\Users\Dell\Aqua_OC_India\MODULE 3\Predicted_POC.tif


In [ ]:
Complete Workflow
Landsat Image

↓

Cloud Mask

↓

Water Mask

↓

NDWI

↓

MNDWI

↓

Feature Stack

↓

Training Samples

↓

Random Forest

↓

Prediction

↓

POC Map

↓

Export GeoTIFF